# Notebook 05 — Striatum Investigation and Gene-Set Robustness

Notebook 04 established the headline PD result: at the ASD-matched circuit-search operating
point, the curated 19-gene `PD_HighConf_DA` set recovers SNc, VTA and 3-4 raphe nuclei at
sizes 11/13/15/20, and the blinded pathology evaluation confirms the model's top-ranked
structures are genuine sites of PD neuropathology. This notebook documents the follow-on
question a careful reader (and Reviewer 2) asks next: **the caudoputamen — the structure
whose degeneration is definitionally "the parkinsonian striatum" — is absent from every one
of those circuits, and ranks 205th of 213 on the curated set's own bias. Is that biology,
curation, or method?**

All of the analysis below previously existed only as scratch scripts in
`notebooks_disease_validation/reference/` (`cherry.py`, `core6.py`, `core6_ccs.py`,
`core6_top30.py`, `why_cp.py`, `cp_conn.py`, `cp_out.py`, `fig2b.py`, `loop_scan.py`,
`front40.py`, `fake_peaks_res.py`, `ot_sweep.py`, `ot_spec.py`, `ot_bias.py`, `ot_circ.py`,
`ot06.py`, `real40res.py`, `striatum.py`, `cp_rank.py`). This notebook is the reproducible,
sectioned consolidation of that work — every number below is recomputed live from files
already on disk, checked against the reference scripts throughout.

**Scope / constraints.**
- `results/` is **read-only** here, with the single exception of `results/PD_HD_validation/`.
  Every Pareto front under `results/CircuitSearch/{PD_HighConf_DA, PD_FAKE_cherrypicked,
  PD_OpenTargets_cut06}/` was produced by the SA circuit search in earlier tasks (many hours
  of compute) — this notebook never re-runs Snakemake, SA search, or the bias pipeline. It
  only ever reads those fronts, and separately performs cheap, non-SA computations (direct
  top-N bias ranking, `ScoreCircuit_SI_Joint` connectivity scoring, sibling-pool FDR draws)
  that take seconds, not hours.
- Two gene sets appear here that are **not** legitimate PD gene sets: a classic 6-gene
  Mendelian panel (a real, if under-powered, comparison set) and a deliberately rigged
  17-gene "cherry-picked" set (a **diagnostic only**, flagged unmistakably in Section 6).

**Contents.** 1. Setup. 2. The question. 3. Per-gene decomposition. 4. Gene-set robustness
(4 independently-derived sets). 5. Why CCS rejects the striatum. 6. The cherry-picking
diagnostic. 7. CCS profiles with sibling nulls. 8. Pareto fronts and the motor loop.
9. Synthesis. 10. Verification gate.

In [ ]:
%load_ext autoreload
%autoreload 2

# 1. Setup

In [ ]:
import collections
import glob
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from adjustText import adjust_text
from joblib import Parallel, delayed

sys.path.insert(1, "../src")
from ASD_Circuits import (
    Fil2Dict,
    GetPermutationP_vectorized,
    LoadGeneINFO,
    MouseSTR_AvgZ_Weighted,
    STR2Region,
    ScoreCircuit_SI_Joint,
)
from plot import REGION_COLORS, bh_fdr, format_pval, pretty_pval_allstyle

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 42
N_JOBS = 10
rng = np.random.default_rng(SEED)

with open("../config/config.yaml") as f:
    config = yaml.safe_load(f)
with open("../config/config.STR.yaml") as f:
    config_str = yaml.safe_load(f)

OUT_DIR = "../results/PD_HD_validation"
FIG_DIR = f"{OUT_DIR}/figures"
TAB_DIR = f"{OUT_DIR}/tables"
CACHE_DIR = f"{OUT_DIR}/cache"
for d in (FIG_DIR, TAB_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)

Reference data: the same Z2 bias matrix, connectome scoring matrices and gene annotation
every other notebook in this series uses, loaded from `config.yaml`/`config.STR.yaml` rather
than hardcoded (`weightmat_ipsi` / `opentargets_pd` were added to `config.yaml`'s
`data_files:` block for this notebook; both are pre-existing static reference data, not
anything this project computes).

In [ ]:
anno = STR2Region()
HGNC, ENSID2Entrez, S2E, E2S = LoadGeneINFO()

Z2 = pd.read_parquet(f"../{config['analysis_types']['STR_ISH']['expr_matrix']}")
InfoMat = pd.read_csv(f"../{config['data_files']['infomat_ipsi']}", index_col=0)
WeightMat = pd.read_csv(f"../{config['data_files']['weightmat_ipsi']}", index_col=0)

M = Z2.values
Z2_ROW = {g: i for i, g in enumerate(Z2.index)}
sibling_pool_df = pd.read_csv(f"../{config['data_files']['sibling_weights']}", header=None)
SIB = np.array([Z2_ROW[g] for g in sibling_pool_df[0].astype(int) if g in Z2_ROW])

print(f"Z2 bias matrix: {Z2.shape[0]} genes x {Z2.shape[1]} structures")
print(f"InfoMat / WeightMat: {InfoMat.shape}")
print(f"Sibling gene pool: {len(sibling_pool_df)} genes, {len(SIB)} present in Z2")

In [ ]:
gw_path = f"../{config_str['gene_sets']['PD_HighConf_DA']['geneweights']}"
genes19 = [int(g) for g in Fil2Dict(gw_path)]
assert len(genes19) == 19, f"PD_HighConf_DA.gw has {len(genes19)} genes, expected 19"

bias_curated = pd.read_csv("../results/STR_ISH/PD_HighConf_DA_bias_addP_random.csv", index_col=0)
bias_curated_sibmut = pd.read_csv(
    "../results/STR_ISH/PD_HighConf_DA_SibMut_bias_addP_sibling.csv", index_col=0)

print(f"PD_HighConf_DA: {len(genes19)} genes -> "
      f"{', '.join(sorted(E2S.get(g, str(g)) for g in genes19))}")

# 2. The Question

The curated 19-gene PD set recovers a clean dopaminergic/monoaminergic circuit (notebook 04).
But the caudoputamen (CP) — where dopaminergic terminal loss actually produces the motor
syndrome, and the structure any clinician would call "the parkinsonian striatum" — never
appears in any recovered circuit at any size. Its bias rank, computed directly below, is why.

In [ ]:
cp_effect = bias_curated.loc["Caudoputamen", "EFFECT"]
cp_rank = int(bias_curated.loc["Caudoputamen", "Rank"])
cp_q_sibmut = bias_curated_sibmut.loc["Caudoputamen", "q-value"]

print(f"Caudoputamen, PD_HighConf_DA (curated, 19 genes):")
print(f"  EFFECT = {cp_effect:.3f}   rank = {cp_rank}/213   q (pipeline SibMut null) = "
      f"{cp_q_sibmut:.4f}  (~{cp_q_sibmut:.2f})")

assert cp_rank == 205, f"curated-set Caudoputamen rank changed: {cp_rank} != 205"
assert round(cp_effect, 3) == -0.515, f"curated-set CP EFFECT changed: {cp_effect:.3f} != -0.515"

Three non-exclusive explanations, tested in turn below:

1. **Biology** — CP genuinely is not where these 19 genes act (Section 3).
2. **Curation** — the 19-gene set is idiosyncratic, and an equally defensible, independently
   derived PD gene set would restore CP (Section 4).
3. **Method** — the CCS connectivity objective structurally cannot select CP regardless of its
   genetic bias, either for a real reason (connectivity) or an artifact of connectome coverage
   (Section 5).

Section 6 then shows what it *would* take to force the striatum in — and that even doing so
on purpose does not produce a defensible or significant circuit. Section 7-8 quantify how
robust the exclusion is across the whole Pareto front and a wider set of circuit sizes.

# 3. Per-Gene Decomposition

`PD_HighConf_DA`'s EFFECT at any structure is the unweighted mean of its 19 genes' Z2 scores
there (`MouseSTR_AvgZ_Weighted` with weight 1.0 for all 19). Averaging can hide a lot — this
section looks at each gene individually at the three structures that matter for the
nigrostriatal story: Caudoputamen (the striatal target), SNc and VTA (the two dopaminergic
source nuclei).

In [ ]:
gene_rows = []
for g in genes19:
    prof = Z2.loc[g]
    cp_rank_within = int(prof.rank(ascending=False)["Caudoputamen"])
    gene_rows.append(dict(
        gene=E2S.get(g, str(g)), entrez=g,
        Z2_CP=prof["Caudoputamen"], CP_rank_within_gene=cp_rank_within,
        Z2_SNc=prof["Substantia_nigra_compact_part"], Z2_VTA=prof["Ventral_tegmental_area"],
    ))
gene_decomp = pd.DataFrame(gene_rows).sort_values("Z2_CP", ascending=False).reset_index(drop=True)
gene_decomp.to_csv(f"{TAB_DIR}/PD_per_gene_CP_SNc_VTA.csv", index=False)

print(f"{'gene':<9s}{'Z2@CP':>9s}{'CP rank/213':>13s}{'Z2@SNc':>10s}{'Z2@VTA':>10s}")
print("-" * 51)
for _, r in gene_decomp.iterrows():
    print(f"{r.gene:<9s}{r.Z2_CP:9.3f}{r.CP_rank_within_gene:13d}{r.Z2_SNc:10.3f}{r.Z2_VTA:10.3f}")

n_pos_cp = int((gene_decomp.Z2_CP > 0).sum())
n_pos_snc = int((gene_decomp.Z2_SNc > 0).sum())
n_pos_vta = int((gene_decomp.Z2_VTA > 0).sum())
print(f"\npositive @ CP: {n_pos_cp}/19   @ SNc: {n_pos_snc}/19   @ VTA: {n_pos_vta}/19")

assert n_pos_cp == 5, f"n positive @ CP changed: {n_pos_cp} != 5"
assert n_pos_snc == 16, f"n positive @ SNc changed: {n_pos_snc} != 16"
assert n_pos_vta == 17, f"n positive @ VTA changed: {n_pos_vta} != 17"

`LRRK2` is the lone outlier that behaves oppositely to the other 18 genes: CP is its single
highest-ranked structure of all 213, while it is one of only 3 genes (of 19) that is
*negative* at SNc.

In [ ]:
lrrk2 = gene_decomp.set_index("gene").loc["LRRK2"]
print(f"LRRK2: Z2@CP = {lrrk2.Z2_CP:+.3f}  (rank {int(lrrk2.CP_rank_within_gene)} of its own "
      f"213-structure profile)   Z2@SNc = {lrrk2.Z2_SNc:+.3f}")

assert lrrk2.CP_rank_within_gene == 1, "LRRK2's own top structure is expected to be CP"
assert lrrk2.Z2_CP > 0 and round(lrrk2.Z2_CP, 2) == 4.93, f"LRRK2 Z2@CP changed: {lrrk2.Z2_CP:.3f}"
assert lrrk2.Z2_SNc < 0 and round(lrrk2.Z2_SNc, 3) == -0.896, \
    f"LRRK2 Z2@SNc changed: {lrrk2.Z2_SNc:.3f}"

## Figure — per-gene Z2 at SNc vs. CP

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 5.2), dpi=100)

is_lrrk2 = gene_decomp.gene == "LRRK2"
ax.axhline(0, color="#999999", lw=0.8, zorder=1)
ax.axvline(0, color="#999999", lw=0.8, zorder=1)
ax.scatter(gene_decomp.loc[~is_lrrk2, "Z2_SNc"], gene_decomp.loc[~is_lrrk2, "Z2_CP"],
           color="#268ad5", s=60, edgecolor="white", linewidth=0.6, zorder=3, label="other 18 genes")
ax.scatter(gene_decomp.loc[is_lrrk2, "Z2_SNc"], gene_decomp.loc[is_lrrk2, "Z2_CP"],
           color="#c0392b", s=110, marker="*", edgecolor="white", linewidth=0.6, zorder=4,
           label="LRRK2")
gene_texts = [ax.text(r.Z2_SNc, r.Z2_CP, r.gene, fontsize=7.5, color="#444444", ha="center")
              for _, r in gene_decomp.iterrows()]
adjust_text(gene_texts, ax=ax, arrowprops=dict(arrowstyle="-", color="#999999", lw=0.5))

ax.set_xlabel("Gene Z2 at Substantia nigra, compact part", fontsize=10)
ax.set_ylabel("Gene Z2 at Caudoputamen", fontsize=10)
ax.set_title(f"PD_HighConf_DA: per-gene bias, SNc vs. CP\n"
             f"{n_pos_cp}/19 positive @ CP, {n_pos_snc}/19 @ SNc — LRRK2 is the sole reversal",
             fontsize=10.5)
ax.legend(fontsize=9, frameon=False, loc="upper right")
ax.grid(alpha=0.25)

fig.patch.set_alpha(0)
ax.patch.set_alpha(0)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/05a_gene_decomposition_CP_vs_SNc.png",
            transparent=True, dpi=300, bbox_inches="tight")
plt.show()

# 4. Gene-Set Robustness

Is CP's exclusion an artifact of curating this *particular* 19-gene set? Four independently
derived PD gene sets, run through the same bias + FDR test, answer that:

- **(a) curated 19** (`PD_HighConf_DA`) — the literature-curated Mendelian panel notebooks
  01-04 use throughout.
- **(b) classic core-6** (`LRRK2 SNCA GBA PRKN PINK1 PARK7`) — the textbook familial-PD gene
  list, independent of notebook 01's curation process.
- **(c) Open Targets PD** (`globalScore >= 0.6`, 47 genes, score-weighted) — an
  algorithmically derived, evidence-scored set from an external database, with no curator
  in the loop at all.
- **(d) cherry-picked 17** — **not an independent set**; a deliberately rigged diagnostic
  built *from* the curated set specifically to force CP in. Constructed and interpreted fully
  in Section 6; previewed here only so the four-way comparison table is in one place.

**Null-model note.** (a)'s FDR (`q-value` below) is the genuine pipeline output — mutability-
weighted sibling sampling via `Snakefile.bias`, the exact procedure the published ASD result
uses. (b)/(c)/(d) were never run through that pipeline (they are not registered gene sets),
so for a like-for-like comparison this notebook applies the *same* ad-hoc diagnostic FDR test
to all three: 10,000 draws of a size- and (for (c)) globalScore-weight-matched gene set,
uniformly at random from the same 1,292-gene sibling-mutation pool `reference/{cherry,
core6}.py` used — not mutability-weighted. Section 7 shows this ad-hoc null is somewhat more
conservative than the pipeline's mutability-weighted null (fewer structures reach q<0.10), so
it is, if anything, a harder bar for (b)/(c)/(d) to clear than (a)'s own pipeline test — not
an easier one.

In [ ]:
def sibling_fdr_test(gene_weights, seed=SEED, n_sims=10000):
    """Bias + BH-FDR against a uniform draw from the sibling-mutation gene pool.

    gene_weights: {entrez: weight}. weight=1.0 for every gene reduces to the unweighted mean
    `cherry.py`/`core6.py` use; arbitrary weights (e.g. Open Targets globalScore) reduce to
    `MouseSTR_AvgZ_Weighted`'s weighted average (verified bit-identical for the observed
    statistic). NOT the pipeline's mutability-weighted null -- see the note above.
    """
    genes = np.array(list(gene_weights.keys()))
    w = np.array([gene_weights[g] for g in genes], dtype=float)
    rows = np.array([Z2_ROW[g] for g in genes])
    obs_sub = M[rows]
    obs_mask = ~np.isnan(obs_sub)
    obs = np.nansum(obs_sub * w[:, None] * obs_mask, axis=0) / np.sum(w[:, None] * obs_mask, axis=0)

    rng_local = np.random.default_rng(seed)
    n = len(genes)
    null = np.empty((n_sims, M.shape[1]))
    for i in range(n_sims):
        dr = rng_local.choice(SIB, size=n, replace=False)
        sub = M[dr]
        mask = ~np.isnan(sub)
        null[i] = np.nansum(sub * w[:, None] * mask, axis=0) / np.sum(w[:, None] * mask, axis=0)

    _, p, _ = GetPermutationP_vectorized(null, obs)
    q = bh_fdr(p)
    out = pd.DataFrame({"EFFECT": obs, "p_sibling": p, "q_sibling": q}, index=Z2.columns)
    out["Rank"] = out["EFFECT"].rank(ascending=False).astype(int)
    return out.sort_values("Rank")

## 4.1 (a) Curated 19 (`PD_HighConf_DA`) — pipeline result

In [ ]:
n_sig_a_q05 = int((bias_curated_sibmut["q-value"] < 0.05).sum())
n_sig_a_q10 = int((bias_curated_sibmut["q-value"] < 0.10).sum())
print(f"curated 19: q<0.05 = {n_sig_a_q05} structures, q<0.10 = {n_sig_a_q10} structures "
      "(pipeline SibMut null)")
for s in ["Caudoputamen", "Substantia_nigra_compact_part", "Ventral_tegmental_area"]:
    print(f"  {s:<34s} rank {int(bias_curated.loc[s,'Rank']):>3d}  EFFECT {bias_curated.loc[s,'EFFECT']:+.3f}"
          f"  q(SibMut) {bias_curated_sibmut.loc[s,'q-value']:.4f}")

## 4.2 (b) Classic core-6 (`LRRK2 SNCA GBA PRKN PINK1 PARK7`)

The textbook familial-PD panel, independent of notebook 01's curation. Cached from
`reference/core6.py` (`results/PD_HD_validation/bundle/PD_core6_structure_bias.csv`, itself
under the writable `PD_HD_validation/` tree) rather than recomputed inline: `core6.py`'s
random+sibling null shares one `np.random.default_rng(42)` object across both pools in
sequence, so a from-scratch reproduction that seeds the sibling draw directly (as
`sibling_fdr_test` above does) lands on a *different*, equally valid but non-identical random
draw — reading the cached file avoids that RNG-order footgun while still being the literal
output of a script in `reference/`, checked below to the numbers this section is built on.

In [ ]:
core6_bias = pd.read_csv(f"{OUT_DIR}/bundle/PD_core6_structure_bias.csv")
n_sig_b_q10 = int((core6_bias.q_sibling < 0.10).sum())
n_sig_b_q05 = int((core6_bias.q_sibling < 0.05).sum())
min_q_b = core6_bias.q_sibling.min()

print(f"core-6: q<0.05 = {n_sig_b_q05} structures, q<0.10 = {n_sig_b_q10} structures  "
      f"(min q_sibling = {min_q_b:.3f})")
for s in ["Caudoputamen", "Substantia_nigra_compact_part", "Ventral_tegmental_area"]:
    r = core6_bias.set_index("structure").loc[s]
    print(f"  {s:<34s} rank {int(r['rank']):>3d}  EFFECT {r.EFFECT:+.3f}  q_sibling {r.q_sibling:.4f}")

assert n_sig_b_q10 == 0, f"core-6 structures at q<0.10 changed: {n_sig_b_q10} != 0"
assert round(min_q_b, 2) == 0.67, f"core-6 min q_sibling changed: {min_q_b:.3f} != ~0.67"
snc_rank_b = int(core6_bias.set_index("structure").loc["Substantia_nigra_compact_part", "rank"])
assert snc_rank_b == 43, f"core-6 SNc rank changed: {snc_rank_b} != 43"

## 4.3 (c) Open Targets PD (`globalScore >= 0.6`)

An algorithmically evidence-scored, externally derived gene list — no curator in the loop.
`EFFECT` uses `MouseSTR_AvgZ_Weighted` with `globalScore` as the per-gene weight (matching
`reference/ot_bias.py`); the FDR null below uses the *same* weights on randomly drawn genes
(`sibling_fdr_test`), so weighting cannot by itself explain a difference from (a)/(b)/(d).

In [ ]:
ot_tsv = pd.read_csv(f"../{config['data_files']['opentargets_pd']}", sep="\t")
ot_tsv["entrez"] = [S2E.get(s) for s in ot_tsv["symbol"]]
ot_tsv = ot_tsv.dropna(subset=["entrez"])
ot_tsv["entrez"] = ot_tsv["entrez"].astype(int)
ot_tsv = ot_tsv[ot_tsv.entrez.isin(Z2.index)]
ot_sel = ot_tsv[ot_tsv.globalScore >= 0.6]
w_ot = dict(zip(ot_sel.entrez, ot_sel.globalScore))
print(f"Open Targets PD, globalScore>=0.6: {len(w_ot)} genes")
assert len(w_ot) == 47, f"OT-PD@0.6 gene count changed: {len(w_ot)} != 47"

ot_bias = sibling_fdr_test(w_ot)
n_sig_c_q10 = int((ot_bias.q_sibling < 0.10).sum())
n_sig_c_q05 = int((ot_bias.q_sibling < 0.05).sum())
print(f"q<0.05 = {n_sig_c_q05} structures, q<0.10 = {n_sig_c_q10} structures  (ad-hoc null)")
for s in ["Caudoputamen", "Substantia_nigra_compact_part", "Ventral_tegmental_area"]:
    r = ot_bias.loc[s]
    print(f"  {s:<34s} rank {int(r.Rank):>3d}  EFFECT {r.EFFECT:+.4f}  q_sibling {r.q_sibling:.4f}")

top5_ot = list(ot_bias.sort_values("Rank").index[:5])
print("top 5:", top5_ot)
rho_ot_curated = ot_bias["EFFECT"].reindex(bias_curated.index).corr(
    bias_curated["EFFECT"], method="spearman")
print(f"Spearman rho vs. curated 19-gene profile = {rho_ot_curated:.3f}")

assert {"Ventral_tegmental_area", "Substantia_nigra_compact_part"} <= set(top5_ot), \
    "expected VTA and SNc in the OT-PD@0.6 top 5"
assert any("raphe" in s.lower() for s in top5_ot), "expected a raphe nucleus in the OT-PD@0.6 top 5"
assert round(rho_ot_curated, 1) == 0.8, f"OT-vs-curated rho changed: {rho_ot_curated:.3f}"

## 4.4 (d) Cherry-picked 17 — preview

Built by removing the 5 most CP-negative genes from the curated set and adding 3 striatal
marker genes that are **not** Parkinson's genes. Full construction rationale, the fact this
is a diagnostic that must never be treated as a real gene set, and the CCS consequence are in
Section 6 — this cell only builds the inputs the Section 4.5 table needs.

In [ ]:
drop5 = gene_decomp.nsmallest(5, "Z2_CP")["entrez"].tolist()
add3 = [int(S2E[s]) for s in ["PDE8B", "ADCY5", "GNAL"]]
fake17 = [g for g in genes19 if g not in drop5] + add3
assert len(fake17) == 17, f"cherry-picked set size changed: {len(fake17)} != 17"

cherry_bias = sibling_fdr_test({g: 1.0 for g in fake17})
n_sig_d_q10 = int((cherry_bias.q_sibling < 0.10).sum())
n_sig_d_q05 = int((cherry_bias.q_sibling < 0.05).sum())
print(f"cherry-picked 17: q<0.05 = {n_sig_d_q05}, q<0.10 = {n_sig_d_q10} structures  "
      f"(preview -- see Section 6 for the full story)")

## 4.5 Combined comparison

In [ ]:
def _row(label, n_genes, null_type, bias_df, rank_col, effect_col, q_col, n10, n05):
    r = dict(gene_set=label, n_genes=n_genes, null_type=null_type,
             n_sig_q10=n10, n_sig_q05=n05)
    for tag, s in [("CP", "Caudoputamen"), ("SNc", "Substantia_nigra_compact_part"),
                   ("VTA", "Ventral_tegmental_area")]:
        r[f"{tag}_rank"] = int(bias_df.loc[s, rank_col])
        r[f"{tag}_EFFECT"] = bias_df.loc[s, effect_col]
        r[f"{tag}_q"] = bias_df.loc[s, q_col]
    return r


robustness = pd.DataFrame([
    _row("(a) curated 19", 19, "pipeline (mutability sibling)",
         bias_curated_sibmut, "Rank", "EFFECT", "q-value", n_sig_a_q10, n_sig_a_q05),
    _row("(b) core-6 classic", 6, "ad-hoc (uniform sibling), cached",
         core6_bias.set_index("structure"), "rank", "EFFECT", "q_sibling", n_sig_b_q10, n_sig_b_q05),
    _row("(c) Open Targets >=0.6", 47, "ad-hoc (uniform sibling)",
         ot_bias, "Rank", "EFFECT", "q_sibling", n_sig_c_q10, n_sig_c_q05),
    _row("(d) cherry-picked [DIAGNOSTIC]", 17, "ad-hoc (uniform sibling)",
         cherry_bias, "Rank", "EFFECT", "q_sibling", n_sig_d_q10, n_sig_d_q05),
])
robustness.to_csv(f"{TAB_DIR}/PD_geneset_robustness_summary.csv", index=False)
with pd.option_context("display.max_columns", None):
    print(robustness[["gene_set", "n_genes", "null_type", "n_sig_q10", "n_sig_q05",
                       "CP_rank", "CP_EFFECT", "CP_q", "SNc_rank", "VTA_rank"]].to_string(index=False))

Three independently derived sets — (a), (b), (c), built by three different people/methods
with no knowledge of each other — all put CP near the bottom of the atlas and SNc/VTA near
the top. (d) is the only set where CP rises, and it is not independent: it exists only
because it was built to do that (Section 6).

## Figure — bias at CP / SNc / VTA across the four gene sets

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.4), dpi=100)

labels = robustness.gene_set.tolist()
x = np.arange(len(labels))
width = 0.25
colors = {"CP": REGION_COLORS["Striatum"], "SNc": REGION_COLORS["Midbrain"], "VTA": "#8e44ad"}
for i, tag in enumerate(["CP", "SNc", "VTA"]):
    ax.bar(x + (i - 1) * width, robustness[f"{tag}_EFFECT"], width=width,
           color=colors[tag], label=tag, edgecolor="white", linewidth=0.6)

ax.axhline(0, color="#555555", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=18, ha="right", fontsize=8.5)
ax.set_ylabel("EFFECT (mean Z2 bias)", fontsize=10)
ax.set_title("Structure bias at CP / SNc / VTA, four independently-derived PD gene sets", fontsize=10.5)
ax.legend(fontsize=9, frameon=False)
ax.grid(axis="y", alpha=0.25)

fig.patch.set_alpha(0)
ax.patch.set_alpha(0)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/05b_geneset_robustness_bias.png",
            transparent=True, dpi=300, bbox_inches="tight")
plt.show()

# 5. Why CCS Rejects the Striatum

`ScoreCircuit_SI_Joint` averages pairwise connectivity information *within* a circuit — it is
a **mean**, not a sum. Adding a structure only raises the score if that structure's pairwise
information with the existing members exceeds the circuit's current average; a well-connected
circuit member can still lower the score if it is poorly connected to *this particular* set.

## 5.1 Adding CP vs. adding STN to the top-13

In [ ]:
core13 = list(bias_curated.sort_values("EFFECT", ascending=False).index[:13])
ccs_top13 = ScoreCircuit_SI_Joint(core13, InfoMat)
ccs_top13_plus_cp = ScoreCircuit_SI_Joint(core13 + ["Caudoputamen"], InfoMat)
ccs_top13_plus_stn = ScoreCircuit_SI_Joint(core13 + ["Subthalamic_nucleus"], InfoMat)

print(f"top-13 alone                      CCS = {ccs_top13:.4f}  (n=13)")
print(f"  + Caudoputamen                  CCS = {ccs_top13_plus_cp:.4f}  "
      f"({ccs_top13_plus_cp - ccs_top13:+.4f})")
print(f"  + Subthalamic_nucleus           CCS = {ccs_top13_plus_stn:.4f}  "
      f"({ccs_top13_plus_stn - ccs_top13:+.4f})")

assert ccs_top13_plus_cp < ccs_top13, "adding CP was expected to LOWER CCS"
assert ccs_top13_plus_stn > ccs_top13, "adding STN was expected to RAISE CCS"
assert round(ccs_top13, 3) == 0.974 and round(ccs_top13_plus_cp, 3) == 0.889, \
    "top-13 / top-13+CP CCS values changed from the reference numbers"

## 5.2 CP's actual connectivity to the top-13 circuit

In [ ]:
conn = [(s, WeightMat.loc["Caudoputamen", s], WeightMat.loc[s, "Caudoputamen"],
         InfoMat.loc["Caudoputamen", s], InfoMat.loc[s, "Caudoputamen"]) for s in core13]
cp_partners = [s for s, wo, wi, *_ in conn if wo > 0 or wi > 0]
mean_info_cp_pairs = np.nanmean([x[3] for x in conn] + [x[4] for x in conn])
within_vals = InfoMat.loc[core13, core13].values
mean_info_within = within_vals[np.nonzero(within_vals)].mean()

print(f"CP connects to {len(cp_partners)}/13 circuit members: {cp_partners}")
print(f"mean info of CP's pairs with the circuit  = {mean_info_cp_pairs:.3f}")
print(f"mean info WITHIN the 13-structure circuit = {mean_info_within:.3f}")

assert cp_partners == ["Substantia_nigra_compact_part"], \
    "CP was expected to connect to exactly SNc among the top-13"
assert round(mean_info_cp_pairs, 3) == 0.376 and round(mean_info_within, 3) == 0.974

## 5.3 Connectome coverage caveat — state this prominently

Part of CP's exclusion is a real biological/topological fact (Section 5.2: it is connected to
only 1 of 13 circuit members). But part of it is a **connectome coverage gap**: the classic
thalamostriatal projection (parafascicular nucleus -> caudoputamen) is entirely absent from
this scoring matrix, and CP's own outputs to the pallidal/nigral output stations are tiny next
to the input it receives from SNc. The exclusion of the striatum is therefore **part biology,
part connectome coverage** — both halves must be reported, not just the first.

In [ ]:
pf_to_cp = WeightMat.loc["Parafascicular_nucleus", "Caudoputamen"]
cp_to_snr = WeightMat.loc["Caudoputamen", "Substantia_nigra_reticular_part"]
cp_to_gpe = WeightMat.loc["Caudoputamen", "Globus_pallidus_external_segment"]
cp_to_gpi = WeightMat.loc["Caudoputamen", "Globus_pallidus_internal_segment"]
snc_to_cp = WeightMat.loc["Substantia_nigra_compact_part", "Caudoputamen"]
snr_to_vmthal = WeightMat.loc["Substantia_nigra_reticular_part",
                               "Ventral_medial_nucleus_of_the_thalamus"]

print(f"Parafascicular_nucleus -> Caudoputamen (thalamostriatal) weight = {pf_to_cp:.3f}  "
      "<== ABSENT from this connectome")
print(f"Caudoputamen -> Substantia_nigra_reticular_part  weight = {cp_to_snr:.3f}")
print(f"Caudoputamen -> Globus_pallidus_external_segment weight = {cp_to_gpe:.3f}")
print(f"Caudoputamen -> Globus_pallidus_internal_segment weight = {cp_to_gpi:.3f}")
print(f"  (for scale) Substantia_nigra_compact_part -> Caudoputamen weight = {snc_to_cp:.3f}")
print(f"Substantia_nigra_reticular_part -> Ventral_medial_nucleus_of_the_thalamus "
      f"weight = {snr_to_vmthal:.3f}  <== ABSENT")

assert pf_to_cp == 0.0, "Parafascicular_nucleus -> Caudoputamen was expected to be absent (0)"
assert snr_to_vmthal == 0.0, "SNr -> VM-thalamus was expected to be absent (0)"
assert cp_to_snr < snc_to_cp and cp_to_gpe < snc_to_cp and cp_to_gpi < snc_to_cp, \
    "CP's outputs were expected to be small relative to its SNc input"

## Figure — CCS response to adding CP vs. STN

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 4.4), dpi=100)

bars = ["top-13\nalone", "+ Caudoputamen\n(genetically ranked\nCP would help)",
        "+ Subthalamic\nnucleus"]
vals = [ccs_top13, ccs_top13_plus_cp, ccs_top13_plus_stn]
bar_colors = ["#7f8c8d", REGION_COLORS["Striatum"], "#2c9d39"]
b = ax.bar(bars, vals, color=bar_colors, edgecolor="white", linewidth=0.8, width=0.65)
for rect, v in zip(b, vals):
    ax.text(rect.get_x() + rect.get_width() / 2, v + 0.02, f"{v:.3f}",
            ha="center", fontsize=9)
ax.axhline(ccs_top13, color="#333333", lw=0.9, ls="--", zorder=1)
ax.set_ylabel("Circuit Connectivity Score (mean pairwise info)", fontsize=9.5)
ax.set_title("CCS is a mean, not a sum:\nCP lowers it, STN raises it", fontsize=10.5)
ax.set_ylim(0, max(vals) * 1.2)
ax.tick_params(labelsize=8.5)
ax.grid(axis="y", alpha=0.25)

fig.patch.set_alpha(0)
ax.patch.set_alpha(0)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/05c_why_CP_rejected_by_CCS.png",
            transparent=True, dpi=300, bbox_inches="tight")
plt.show()

# 6. The Cherry-Picking Diagnostic

⚠️ **This gene set is a diagnostic probe, constructed deliberately to force the striatum into
the ranking. It must never be reported, cited, or reused as a legitimate Parkinson's gene
set.** It exists to answer one question: if CP's exclusion were purely a curation artifact,
could a plausible-looking rescue set fix it? The answer is no, even when the rescue is rigged.

**Construction.** Starting from the curated 19, drop the 5 most CP-negative genes (Section
3's `nsmallest(5, Z2_CP)`: `DNAJC6`, `ATP13A2`, `GBA`, `SYNJ1`, `PLA2G6`) and add 3 genes whose
mouse expression profile is dominated by the striatum but whose human phenotype has nothing to
do with Parkinson's neurodegeneration: `PDE8B` (autosomal-dominant striatal degeneration with
prominent dystonia), `ADCY5` (ADCY5-related dyskinesia), `GNAL` (dystonia/chorea). None of the
three is a Parkinson's gene; they are chosen purely for their striatal expression footprint.

In [ ]:
print(f"dropped (5 most CP-negative of the 19): {[E2S.get(g) for g in drop5]}")
print(f"added (3 striatal markers, not PD genes): {[E2S.get(g) for g in add3]}")
print(f"cherry-picked set (n={len(fake17)}): {sorted(E2S.get(g, str(g)) for g in fake17)}")

## 6.1 Does it work? Only partially, and it still fails FDR.

In [ ]:
cp_rank_cherry = int(cherry_bias.loc["Caudoputamen", "Rank"])
nacc_rank_cherry = int(cherry_bias.loc["Nucleus_accumbens", "Rank"])
cp_q_cherry = cherry_bias.loc["Caudoputamen", "q_sibling"]

print("top 15 of 213, cherry-picked set:")
for s, r in cherry_bias.sort_values("Rank").head(15).iterrows():
    flag = "  <== STRIATUM" if anno.get(s) in ("Striatum", "Pallidum") else ""
    print(f"  {int(r.Rank):>3d}. EFFECT {r.EFFECT:+.3f}  q {r.q_sibling:.3f}  {s:<40s}{flag}")

print(f"\nCaudoputamen:      rank {cp_rank_cherry}/213  (was {cp_rank}/213 in the curated set)")
print(f"Nucleus_accumbens: rank {nacc_rank_cherry}/213")
print(f"CP q_sibling = {cp_q_cherry:.4f}  -- {'FAILS' if cp_q_cherry >= 0.05 else 'passes'} q<0.05")
print(f"structures at q<0.10: curated {n_sig_a_q10} (pipeline) -> cherry-picked {n_sig_d_q10} (ad-hoc)")

assert cp_rank_cherry == 11 and nacc_rank_cherry == 9, \
    f"cherry-picked CP/NAcc ranks changed: {cp_rank_cherry}/{nacc_rank_cherry} != 11/9"
assert cp_q_cherry >= 0.05, "cherry-picked CP was expected to still fail q<0.05"
assert round(cp_q_cherry, 3) == 0.104, f"cherry-picked CP q changed: {cp_q_cherry:.3f} != 0.104"
assert n_sig_d_q10 == 7, f"cherry-picked n_sig q<0.10 changed: {n_sig_d_q10} != 7"

## 6.2 CCS significance collapses

Mirrors `reference/cherry.py`'s CCS block exactly: rank the cherry-picked set's own bias
profile, score `ScoreCircuit_SI_Joint` at a grid of sizes, and test against 3,000 sibling-pool
null draws of the same size (17).

In [ ]:
CHERRY_SIZES = [10, 13, 20, 30, 46, 60]
cherry_order = pd.Series(Z2.loc[fake17].mean(axis=0), index=Z2.columns) \
    .sort_values(ascending=False).index.values


def _ccs_profile(structs, sizes):
    return [ScoreCircuit_SI_Joint(structs[:n], InfoMat) for n in sizes]


cherry_ccs_obs = np.array(_ccs_profile(cherry_order, CHERRY_SIZES))
cols = np.array(Z2.columns)
rng_ccs = np.random.default_rng(7)
cherry_ccs_draws = [rng_ccs.choice(SIB, size=len(fake17), replace=False) for _ in range(3000)]
cherry_ccs_null = np.array(Parallel(n_jobs=N_JOBS)(
    delayed(lambda dr: _ccs_profile(cols[np.argsort(-np.nanmean(M[dr], axis=0))], CHERRY_SIZES))(d)
    for d in cherry_ccs_draws))

cherry_ccs_p = np.array([(np.sum(cherry_ccs_null[:, j] >= cherry_ccs_obs[j]) + 1) / 3001
                          for j in range(len(CHERRY_SIZES))])
n_ccs_sig = int((cherry_ccs_p < 0.05).sum())
for n, ccs, p in zip(CHERRY_SIZES, cherry_ccs_obs, cherry_ccs_p):
    print(f"  N={n:>3d}  CCS {ccs:.3f}  p={p:.4f}{'  <== sig' if p < 0.05 else ''}")
print(f"\n{n_ccs_sig}/{len(CHERRY_SIZES)} sizes reach p<0.05 -- CCS significance largely collapses "
      "for the cherry-picked set (one borderline exception at N=46, p={:.3f}, not the diffuse, "
      "multi-size significance the curated set shows in Section 7).".format(cherry_ccs_p[4]))

assert n_ccs_sig <= 1, "expected CCS significance to have collapsed for the cherry-picked set"

## Figure — CP/NAcc rank shift and significant-structure count, curated vs. cherry-picked

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.4), dpi=100)

ax = axes[0]
label_dy = {"Caudoputamen": -11, "Nucleus_accumbens": 5}
for s, marker, color in [("Caudoputamen", "o", REGION_COLORS["Striatum"]),
                          ("Nucleus_accumbens", "s", "#c0783a")]:
    before = int(bias_curated.loc[s, "Rank"])
    after = int(cherry_bias.loc[s, "Rank"])
    ax.plot([0, 1], [before, after], color=color, lw=1.6, zorder=2)
    ax.scatter([0, 1], [before, after], color=color, marker=marker, s=70, zorder=3,
               edgecolor="white", linewidth=0.6, label=s.replace("_", " "))
    ax.annotate(f"{before}", (0, before), textcoords="offset points",
                xytext=(-8, label_dy[s]), fontsize=8)
    ax.annotate(f"{after}", (1, after), textcoords="offset points",
                xytext=(8, label_dy[s]), fontsize=8)
ax.set_xticks([0, 1])
ax.set_xticklabels(["curated 19\n(real gene set)", "cherry-picked 17\n(rigged diagnostic)"], fontsize=8.5)
ax.set_ylabel("Rank (1 = most implicated, 213 = least)", fontsize=9)
ax.invert_yaxis()
ax.set_title("Striatal structures rise\nwhen deliberately rigged", fontsize=10)
ax.legend(fontsize=7.5, frameon=False, loc="center right")
ax.grid(alpha=0.25)

ax2 = axes[1]
sets2 = ["curated 19\n(pipeline q)", "cherry-picked 17\n(ad-hoc q, rigged)"]
sig10 = [n_sig_a_q10, n_sig_d_q10]
ax2.bar(sets2, sig10, color=["#268ad5", "#c0392b"], edgecolor="white", linewidth=0.8, width=0.55)
for i, v in enumerate(sig10):
    ax2.text(i, v + 0.3, str(v), ha="center", fontsize=10)
ax2.set_ylabel("Structures at q < 0.10", fontsize=9)
ax2.set_title("...but overall significance\nstill drops", fontsize=10)
ax2.grid(axis="y", alpha=0.25)

fig.patch.set_alpha(0)
for a in axes:
    a.patch.set_alpha(0)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/05d_cherrypick_diagnostic.png",
            transparent=True, dpi=300, bbox_inches="tight")
plt.show()

# 7. CCS Profiles with Sibling Nulls

Figure-2B-style view: `ScoreCircuit_SI_Joint` of the top-N structures, N swept broadly, tested
against a sibling-derived null at each size. Two null estimates for the curated set exist and
**must not be conflated** — read the note after the primary result before interpreting the
figure.

## 7.1 Primary result — the pipeline's mutability-weighted null

`results/STR_ISH/null_bias/PD_HighConf_DA_SibMut_null_bias_sibling.parquet` is genuine
`Snakefile.bias` output: 10,000 mutability-weighted sibling gene sets, the exact null model
the published ASD circuit result uses. `reference/ccs_sweep.py` already scored
`ScoreCircuit_SI_Joint` at N=6..70 for the observed ranking and every null column, cached at
`results/PD_HD_validation/bundle/PD_CCS_by_circuit_size.csv` (read here, not recomputed —
rebuilding it would repeat 65 x 10,000 `ScoreCircuit_SI_Joint` calls for no new information).

In [ ]:
ccs_pipeline = pd.read_csv(f"{OUT_DIR}/bundle/PD_CCS_by_circuit_size.csv")
n_sizes_pipeline = len(ccs_pipeline)
n_sig_pipeline = int((ccs_pipeline.p_sibling < 0.05).sum())
p13_pipeline = ccs_pipeline.loc[ccs_pipeline.N == 13, "p_sibling"].iloc[0]

print(f"pipeline (mutability-weighted sibling) null, N=6..{ccs_pipeline.N.max()} "
      f"({n_sizes_pipeline} sizes):")
print(f"  significant at p<0.05: {n_sig_pipeline}/{n_sizes_pipeline} sizes")
print(f"  p at N=13: {p13_pipeline:.4f}")

assert n_sig_pipeline == 44 and n_sizes_pipeline == 65, \
    f"pipeline-null significant-size count changed: {n_sig_pipeline}/{n_sizes_pipeline} != 44/65"
assert round(p13_pipeline, 4) == 0.0048, f"pipeline-null p@N=13 changed: {p13_pipeline:.4f} != 0.0048"

## 7.2 NULL DISCREPANCY — read before interpreting the figure below

The Figure-2B-style plot needs a **percentile band** (15.9-84.1 pct) at every size, which the
cached pipeline summary above does not carry (it stores only the median and the p-value, to
keep the cache small). Rebuilding a banded null the same way `reference/fig2b.py` did —
drawing fresh gene sets **uniformly** from the sibling-mutation pool (*not* mutability-
weighted, unlike Section 7.1) — gives a visibly different answer: **24/75 sizes significant,
p=0.0095 at N=13**, against the pipeline's **44/65 significant, p=0.0048 at N=13**. Both
numbers are real and reproducible; they measure different things (uniform vs. mutability-
weighted sibling sampling) and are never interchangeable. **The pipeline number (7.1) is the
one that matches the published ASD procedure and is the primary claim; the banded figure below
uses the ad-hoc uniform draw only because it is the one that produces a band, and is labelled
as such throughout.**

## 7.3 Ad-hoc banded null (for the figure), curated vs. cherry-picked

Cached under `results/PD_HD_validation/cache/` (the `.npz` pattern notebook 02 already
established) — 2 gene sets x 4,000 sibling draws x 75 sizes of `ScoreCircuit_SI_Joint` takes
~40s uncached; nothing here is a circuit search.

In [ ]:
FIG2B_SIZES = list(range(6, 81))
FIG2B_SETS = {"PD_HighConf_DA": (genes19, bias_curated["EFFECT"]),
              "PD_FAKE_cherrypicked": (fake17, pd.Series(Z2.loc[fake17].mean(axis=0), index=Z2.columns))}


def _null_band(n_genes, nsim=4000, seed=SEED):
    rng_band = np.random.default_rng(seed)
    draws = [rng_band.choice(SIB, size=n_genes, replace=False) for _ in range(nsim)]
    return np.array(Parallel(n_jobs=N_JOBS)(delayed(
        lambda dr: _ccs_profile(cols[np.argsort(-np.nanmean(M[dr], axis=0))], FIG2B_SIZES))(d)
        for d in draws))


band_cache_fp = f"{CACHE_DIR}/nullbands_curated_vs_cherry_seed{SEED}.npz"
band_keys = [f"{lbl}__{stat}" for lbl in FIG2B_SETS for stat in ("CCS", "p", "lo", "hi", "med")]
band_data = {}
if os.path.exists(band_cache_fp):
    _c = np.load(band_cache_fp)
    band_data = {k: _c[k] for k in _c.files}
    if set(band_data) != set(band_keys):
        print("cache key mismatch -- recomputing")
        band_data = {}
    else:
        print(f"loaded cached null bands from {band_cache_fp}")

if not band_data:
    for lbl, (genes, _) in FIG2B_SETS.items():
        obs_profile = np.array(_ccs_profile(
            Z2.loc[genes].mean(axis=0).sort_values(ascending=False).index.values, FIG2B_SIZES))
        nb = _null_band(len(genes))
        band_data[f"{lbl}__CCS"] = obs_profile
        band_data[f"{lbl}__p"] = np.array([(np.sum(nb[:, j] >= obs_profile[j]) + 1) / (nb.shape[0] + 1)
                                            for j in range(len(FIG2B_SIZES))])
        band_data[f"{lbl}__lo"] = np.percentile(nb, 15.9, axis=0)
        band_data[f"{lbl}__hi"] = np.percentile(nb, 84.1, axis=0)
        band_data[f"{lbl}__med"] = np.median(nb, axis=0)
    np.savez_compressed(band_cache_fp, **band_data)
    print(f"computed + cached null bands -> {band_cache_fp}")

Cross-check against `results/PD_HD_validation/exploratory/CCS_profile_real_vs_fake.csv`
(`reference/fig2b.py`'s own cached output) — confirms this notebook's independent
recomputation reproduces the reference script's numbers.

In [ ]:
adhoc_ref = pd.read_csv(f"{OUT_DIR}/exploratory/CCS_profile_real_vs_fake.csv")
p_curated_adhoc = band_data["PD_HighConf_DA__p"]
p_cherry_adhoc = band_data["PD_FAKE_cherrypicked__p"]
n_sig_curated_adhoc = int((p_curated_adhoc < 0.05).sum())
n_sig_cherry_adhoc = int((p_cherry_adhoc < 0.05).sum())
p13_curated_adhoc = p_curated_adhoc[FIG2B_SIZES.index(13)]

max_diff_ccs = np.max(np.abs(band_data["PD_HighConf_DA__CCS"] - adhoc_ref.CCS_PD_HighConf_DA.values))
max_diff_p = np.max(np.abs(p_curated_adhoc - adhoc_ref.p_PD_HighConf_DA.values))
print(f"cross-check vs. reference/fig2b.py's cached output: "
      f"max|CCS diff|={max_diff_ccs:.2e}  max|p diff|={max_diff_p:.2e}")

print(f"\nad-hoc uniform-sibling null, N=6..80 (75 sizes):")
print(f"  curated 19:        significant p<0.05: {n_sig_curated_adhoc}/75   p@N=13 = {p13_curated_adhoc:.4f}")
print(f"  cherry-picked 17:  significant p<0.05: {n_sig_cherry_adhoc}/75   "
      f"best p = {p_cherry_adhoc.min():.4f} at N={FIG2B_SIZES[int(np.argmin(p_cherry_adhoc))]}")

assert max_diff_ccs < 1e-6 and max_diff_p < 1e-6, "recomputed ad-hoc null diverges from fig2b.py's cache"
assert n_sig_curated_adhoc == 24, f"ad-hoc curated significant-size count changed: {n_sig_curated_adhoc} != 24"
assert round(p13_curated_adhoc, 4) == 0.0095, f"ad-hoc curated p@N=13 changed: {p13_curated_adhoc:.4f} != 0.0095"

## Figure — CCS vs. circuit size, curated vs. cherry-picked, ad-hoc sibling null bands

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.3), dpi=100)
colors_fig2b = {"PD_HighConf_DA": "#1f77b4", "PD_FAKE_cherrypicked": "#c0392b"}
display_lbl = {"PD_HighConf_DA": "curated 19 (real gene set)",
               "PD_FAKE_cherrypicked": "cherry-picked 17 (diagnostic, not a real gene set)"}

for lbl in FIG2B_SETS:
    ccs = band_data[f"{lbl}__CCS"]
    p = band_data[f"{lbl}__p"]
    lo, hi, med = band_data[f"{lbl}__lo"], band_data[f"{lbl}__hi"], band_data[f"{lbl}__med"]
    color = colors_fig2b[lbl]
    ax.plot(FIG2B_SIZES, ccs, color=color, lw=1.8, marker="o", ms=2.5, label=display_lbl[lbl])
    ax.fill_between(FIG2B_SIZES, lo, hi, color=color, alpha=0.13, lw=0)
    ax.plot(FIG2B_SIZES, med, color=color, ls="--", lw=1, alpha=0.6)
    sig_sizes = np.array(FIG2B_SIZES)[p < 0.05]
    ax.plot(sig_sizes, ccs[p < 0.05], "o", ms=6, mfc="none", mec=color, mew=1.6)

ax.axvline(13, color="#888888", lw=0.8, ls=":", zorder=1)
ax.set_xlabel("Number of top-ranked structures (circuit size)", fontsize=11)
ax.set_ylabel("Circuit Connectivity Score", fontsize=11)
ax.set_title("CCS vs. circuit size, ad-hoc uniform-sibling null bands (15.9-84.1 pct)\n"
             "open circles = p<0.05  |  see Section 7.2 -- pipeline null gives 44/65 sig, not 24/75",
             fontsize=9.5)
ax.grid(alpha=0.3, ls="--")
ax.legend(fontsize=9, loc="upper right", frameon=False)

fig.patch.set_alpha(0)
ax.patch.set_alpha(0)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/05e_CCS_vs_size_nullbands.png",
            transparent=True, dpi=300, bbox_inches="tight")
plt.show()

# 8. Pareto Fronts and the Motor Loop

Everything above is bias-level or a single top-N ranking. This section reads the actual SA
circuit-search output (`results/CircuitSearch/`, read-only, already on disk) to ask whether
the striatum re-enters once the search is free to trade bias for connectivity.

In [ ]:
circuit_configs = ["circuit_config_disease.yaml", "circuit_config_disease_size40.yaml",
                    "circuit_config_fake_diagnostic.yaml", "circuit_config_fake_peaks.yaml",
                    "circuit_config_ot_exploratory.yaml"]
input_str_bias = {}
output_dirs = set()
for fname in circuit_configs:
    with open(f"../config/{fname}") as f:
        cfg = yaml.safe_load(f)
    output_dirs.add(cfg["output_dir"])
    for name, spec in cfg["Input_str_bias"].items():
        if name in input_str_bias:
            assert input_str_bias[name] == spec["bias_df"], f"{name}: bias_df path disagrees across configs"
        input_str_bias[name] = spec["bias_df"]
assert len(output_dirs) == 1, "circuit-search configs disagree on output_dir"
CIRCUIT_OUT_DIR = f"../{output_dirs.pop()}"
print(f"circuit datasets registered: {list(input_str_bias)}")
print(f"circuit output dir: {CIRCUIT_OUT_DIR}")


def load_front(dataset, size):
    f = f"{CIRCUIT_OUT_DIR}/{dataset}/pareto_fronts/{dataset}_size_{size}_pareto_front.csv"
    df = pd.read_csv(f)
    assert (df.circuit_type == "baseline").sum() == 1
    return df


def select_at_bias_sacrifice(front, target_pct=-20.0):
    """ASD-matched operating point, mirrors `reference/front40.py`'s exact convention.

    `opt` is re-sorted by mean_bias descending and re-indexed 0..len(opt)-1 *before* picking
    the closest-to-target_pct row, so the returned index is well-defined and comparable across
    fronts (this differs from notebook 04's `select_at_bias_sacrifice`, which never reports an
    index and so never sorts first -- the selected row's structures/CCS/bias are identical
    either way, only the index label depends on this choice).
    """
    base = front.loc[front.circuit_type == "baseline"].iloc[0]
    opt = front.loc[front.circuit_type == "optimized"].copy() \
        .sort_values("mean_bias", ascending=False).reset_index(drop=True)
    opt["dBias_pct"] = 100 * (opt.mean_bias - base.mean_bias) / base.mean_bias
    opt["dCCS_pct"] = 100 * (opt.circuit_score - base.circuit_score) / base.circuit_score
    sel_i = int((opt.dBias_pct - target_pct).abs().argmin())
    return base, opt.iloc[sel_i], sel_i, opt

## 8.1 Size-40 curated front — is the ASD-matched point an extreme?

In [ ]:
front40 = load_front("PD_HighConf_DA", 40)
base40, sel40, sel40_i, opt40 = select_at_bias_sacrifice(front40)
n_opt40 = len(opt40)

print(f"PD_HighConf_DA, size 40: {n_opt40} optimized Pareto points (+ 1 baseline)")
print(f"baseline (top-40 by bias): CCS {base40.circuit_score:.4f}  bias {base40.mean_bias:.4f}")
print(f"ASD-matched selection: index {sel40_i} of {n_opt40 - 1}  "
      f"(0 = highest bias/lowest CCS, {n_opt40 - 1} = lowest bias/highest CCS)")
print(f"  dBias = {sel40.dBias_pct:+.1f}%   dCCS = {sel40.dCCS_pct:+.1f}%   "
      f"CCS = {sel40.circuit_score:.4f}   bias = {sel40.mean_bias:.4f}")
print(f"  is it an extreme? left-most (idx 0) dCCS={opt40.dCCS_pct.iloc[0]:+.1f}%, "
      f"right-most (idx {n_opt40-1}) dCCS={opt40.dCCS_pct.iloc[-1]:+.1f}%  "
      f"-> the selected point sits well inside that range")

assert sel40_i == 14 and n_opt40 == 45, f"ASD-matched index changed: {sel40_i} of {n_opt40-1} != 14 of 44"
assert round(sel40.dBias_pct, 1) == -19.9 and round(sel40.dCCS_pct, 1) == 100.7, \
    "ASD-matched size-40 dBias/dCCS changed from the reference numbers"
assert not (0 < sel40_i < 3) and not (n_opt40 - 4 < sel40_i < n_opt40), \
    "ASD-matched point should not sit at either extreme of the front"

## 8.2 Loop-component scan across ALL Pareto points, curated vs. cherry-picked

Not just the selected point — every optimized point on the size-40 front, for both datasets.
8 canonical basal-ganglia motor-loop components: SNc, SNr, CP, GPe, GPi, STN, VM-thalamus, M1.

In [ ]:
LOOP = {"SNc": "Substantia_nigra_compact_part", "SNr": "Substantia_nigra_reticular_part",
        "CP": "Caudoputamen", "GPe": "Globus_pallidus_external_segment",
        "GPi": "Globus_pallidus_internal_segment", "STN": "Subthalamic_nucleus",
        "VM-thal": "Ventral_medial_nucleus_of_the_thalamus", "M1": "Primary_motor_area"}

loop_scan = {}
for tag, dataset in [("curated (real)", "PD_HighConf_DA"), ("cherry-picked (diagnostic)", "PD_FAKE_cherrypicked")]:
    front = load_front(dataset, 40)
    opt = front.loc[front.circuit_type == "optimized"]
    rows = []
    for _, r in opt.iterrows():
        mem = set(r.structures.split(","))
        rows.append({k: (v in mem) for k, v in LOOP.items()})
    t = pd.DataFrame(rows)
    loop_scan[tag] = t
    print(f"\n{tag}, size 40: {len(t)} optimized points")
    print(f"  max components reached in any single point: {t.sum(axis=1).max()}/8  "
          f"({int((t.sum(axis=1) == t.sum(axis=1).max()).sum())} points)")
    for k in LOOP:
        print(f"    {k:<8s} {int(t[k].sum()):>3d}/{len(t)}")

curated_loop = loop_scan["curated (real)"]
cherry_loop = loop_scan["cherry-picked (diagnostic)"]
curated_max = int(curated_loop.sum(axis=1).max())
cherry_max = int(cherry_loop.sum(axis=1).max())

assert curated_max == 5, f"curated max loop components changed: {curated_max} != 5"
assert cherry_max == 7, f"cherry-picked max loop components changed: {cherry_max} != 7"
for k in ("CP", "GPe", "GPi"):
    assert curated_loop[k].sum() == 0, f"curated set unexpectedly contains {k} in the size-40 front"
assert curated_loop["STN"].sum() == len(curated_loop), "STN was expected in every curated size-40 point"
assert curated_loop["VM-thal"].sum() == len(curated_loop) - 1
assert curated_loop["M1"].sum() == len(curated_loop) - 3
assert cherry_loop["GPi"].sum() == 0, "cherry-picked set unexpectedly contains GPi in the size-40 front"

CP, GPe and GPi appear in **zero** of the curated set's 45 size-40 Pareto points — this holds
across the *entire* trade-off curve, not just the ASD-matched point. STN, VM-thalamus and M1
are in nearly every point instead. The cherry-picked (rigged) set does noticeably better —
7/8 components in its richest point, and CP itself appears in a majority of points — but even
it never recovers GPi, the basal ganglia's principal output nucleus.

## 8.3 What the curated size-40 circuit recovers instead

In [ ]:
sel40_structs = set(sel40.structures.split(","))
BG_LOOP_SET = {"Substantia_nigra_compact_part", "Substantia_nigra_reticular_part",
               "Pedunculopontine_nucleus", "Subthalamic_nucleus",
               "Ventral_medial_nucleus_of_the_thalamus", "Primary_motor_area", "Secondary_motor_area"}
STRIATOPALLIDAL = {"Caudoputamen", "Nucleus_accumbens", "Globus_pallidus_external_segment",
                    "Globus_pallidus_internal_segment"}

present = BG_LOOP_SET & sel40_structs
absent = STRIATOPALLIDAL - sel40_structs
print("present (nigra -> subthalamic -> thalamus -> cortex arc):")
print(" ", sorted(present))
print("absent (the striatopallidal limb):")
print(" ", sorted(absent))
print("\nfull region composition of the selected 40-structure circuit:")
print(" ", dict(collections.Counter(anno.get(s, "?") for s in sel40_structs)))

assert present == BG_LOOP_SET, "expected the full nigra-STN-thalamus-cortex arc in the size-40 circuit"
assert absent == STRIATOPALLIDAL, "expected the full striatopallidal limb to be absent"

## 8.4 Open Targets circuits (sizes 13/40) — does an independent gene set change this?

In [ ]:
CORE_STRIATOPALLIDAL = STRIATOPALLIDAL | {"Fundus_of_striatum"}
for size in (13, 40):
    front = load_front("PD_OpenTargets_cut06", size)
    base, sel, sel_i, opt = select_at_bias_sacrifice(front)
    mem = set(sel.structures.split(","))
    core_hit = mem & CORE_STRIATOPALLIDAL
    broad_hit = {s for s in mem if anno.get(s) in ("Striatum", "Pallidum")}
    print(f"OT-PD size {size:>2d}: idx {sel_i}/{len(opt)-1}  dBias {sel.dBias_pct:+.1f}%  "
          f"CCS {sel.circuit_score:.3f}")
    print(f"  core striatum/pallidum (CP, NAcc, FS, GPe, GPi): {sorted(core_hit) or 'NONE'}")
    if broad_hit - core_hit:
        print(f"  other Striatum/Pallidum-division structures present: {sorted(broad_hit - core_hit)} "
              "(basal-forebrain/septal, not the caudoputamen-GPe-GPi loop)")
    assert not core_hit, f"OT-PD size {size}: unexpected core striatopallidal member {core_hit}"

Neither Open Targets circuit contains the caudoputamen, nucleus accumbens, fundus of striatum,
GPe or GPi at either size. The size-40 circuit does contain one `Pallidum`-division structure,
`Diagonal_band_nucleus` — a basal-forebrain/septal cholinergic nucleus that `STR2Region()`
groups with pallidum alongside the septal complex, anatomically and functionally distinct from
the caudoputamen-GPe-GPi loop this section is about. With that one nuance stated plainly, a
third, independently derived gene set also excludes the striatum from its circuit search.

## Figure — size-40 Pareto front and loop-component coverage

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6.4, 6.4), dpi=100)

ax = axes[0]
opt40_plot = opt40.sort_values("mean_bias")
extreme40 = opt40.loc[opt40.circuit_score.idxmax()]
ax.plot(opt40_plot.circuit_score, opt40_plot.mean_bias, "-", color="#542788", lw=1.5, zorder=2,
        label="Pareto front")
ax.scatter([base40.circuit_score], [base40.mean_bias], marker="D", s=55, color="black", zorder=5,
           label="Baseline (top-40 by bias)")
ax.scatter([sel40.circuit_score], [sel40.mean_bias], marker="x", s=90, color="red", linewidth=2.2,
           zorder=6, label="ASD-matched (idx 14/44)")
ax.scatter([extreme40.circuit_score], [extreme40.mean_bias], marker="o", s=45, facecolor="none",
           edgecolor="#888888", linewidth=1.3, zorder=5, label="Extreme end (max CCS)")
ax.set_xlabel("Circuit Connectivity Score", fontsize=9.5)
ax.set_ylabel("Mean structure bias", fontsize=9.5)
ax.set_title("PD_HighConf_DA, size 40:\nASD-matched point is not an extreme", fontsize=10)
ax.legend(fontsize=7.5, frameon=False, loc="upper right")
ax.grid(alpha=0.25)

ax2 = axes[1]
comp_labels = list(LOOP.keys())
x = np.arange(len(comp_labels))
width = 0.36
curated_frac = [curated_loop[k].mean() for k in comp_labels]
cherry_frac = [cherry_loop[k].mean() for k in comp_labels]
ax2.bar(x - width / 2, curated_frac, width=width, color="#1f77b4", edgecolor="white",
        linewidth=0.6, label="curated 19 (real)")
ax2.bar(x + width / 2, cherry_frac, width=width, color="#c0392b", edgecolor="white",
        linewidth=0.6, label="cherry-picked 17 (diagnostic)")
ax2.set_xticks(x)
ax2.set_xticklabels(comp_labels, rotation=30, ha="right", fontsize=8.5)
ax2.set_ylabel("Fraction of size-40 Pareto points containing component", fontsize=8.5)
ax2.set_title("Motor-loop coverage across\nALL Pareto points", fontsize=10)
ax2.legend(fontsize=7.5, frameon=False)
ax2.grid(axis="y", alpha=0.25)

fig.patch.set_alpha(0)
for a in axes:
    a.patch.set_alpha(0)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/05f_pareto_size40_and_loop_components.png",
            transparent=True, dpi=300, bbox_inches="tight")
plt.show()

# 9. Synthesis

Three independently derived Parkinson's gene sets — the curated 19-gene panel, the textbook
core-6 familial genes, and an algorithmic Open Targets pull with no curator in the loop —
**all** put the caudoputamen near the bottom of the 213-structure atlas and exclude it from
every recovered circuit at every size tested. The one gene set that does not is a diagnostic
built specifically, and admittedly, to force it in — and even that rigged set still fails FDR
(q=0.104) and loses most of its CCS significance. Curation is not the explanation.

The genetics and the connectivity objective independently reject the striatum for different
reasons. Genetically, only 5/19 curated genes are CP-positive, against 16-17/19 for the two
dopaminergic source nuclei (Section 3) — CP's bias is a real, if noisy, negative signal, not
an artifact of one or two outlier genes (LRRK2 is the only reversal, and even it does not
flip the set-level average). Topologically, CCS is a mean, and CP is connected to only 1 of
the top-13 circuit's 13 members — adding it lowers the score exactly as the sparse-connectivity
math predicts (Section 5). But that second reason is not purely biological: the classical
thalamostriatal (parafascicular -> caudoputamen) and nigrothalamic (SNr -> VM-thalamus)
projections are both entirely absent from this connectome, so some of CP's poor connectivity
score reflects matrix coverage, not anatomy. **Both halves — genetic depletion at CP, and a
real-but-incompletely-covered connectome — belong in the rebuttal, not just the first.**

What the circuit search finds *instead* is coherent and specific: the nigra -> subthalamic ->
thalamus -> cortex arc (SNc, SNr, PPN, STN, VM-thalamus, M1, M2), missing only the
striatopallidal limb (CP, NAcc, GPe, GPi) — not a random or anatomically nonspecific
substitute. Every independent line of evidence in this notebook (bias, FDR, CCS at the
selected point, CCS across the whole Pareto front, an independent gene set's own circuit
search) converges on the same boundary.

# 10. Verification Gate

The numbers this section of the rebuttal rests on, restated and re-asserted explicitly so a
change to any upstream file causes a loud failure here.

In [ ]:
assert cp_rank == 205, "curated-set Caudoputamen rank changed"
assert n_sig_b_q10 == 0, "core-6 structures at q<0.10 changed"
assert ccs_top13_plus_cp < ccs_top13 < ccs_top13_plus_stn, \
    "adding CP should lower CCS and adding STN should raise it, relative to the top-13 alone"
assert pf_to_cp == 0.0, "Parafascicular_nucleus -> Caudoputamen weight changed"
assert curated_loop["CP"].sum() == 0 and curated_loop["GPe"].sum() == 0 and curated_loop["GPi"].sum() == 0, \
    "CP/GPe/GPi should be in 0 of the curated set's size-40 Pareto points"
assert sel40_i == 14, "the ASD-matched size-40 point should be index 14"

print("VERIFIED:")
print(f"  curated-set Caudoputamen rank                = {cp_rank}  (== 205)")
print(f"  core-6 structures at q_sibling<0.10           = {n_sig_b_q10}  (== 0)")
print(f"  CCS: top-13 alone / +CP / +STN                = "
      f"{ccs_top13:.4f} / {ccs_top13_plus_cp:.4f} / {ccs_top13_plus_stn:.4f}  "
      "(+CP lowers, +STN raises)")
print(f"  Parafascicular_nucleus -> Caudoputamen weight  = {pf_to_cp:.3f}  (== 0)")
print(f"  CP / GPe / GPi in curated size-40 Pareto front = "
      f"{int(curated_loop['CP'].sum())}/{int(curated_loop['GPe'].sum())}/{int(curated_loop['GPi'].sum())} "
      f"of {n_opt40}  (all == 0)")
print(f"  ASD-matched point index, size-40 curated front = {sel40_i} of {n_opt40 - 1}  (== 14 of 44)")